# Week 6 — Data Cleaning: String Cleaning in Pandas
## Phase 2a Python | PORA Academy Cohort 7

By the end of this session, you will be able to:
- Use the .str accessor for vectorised string operations
- Apply .str.strip(), .str.title(), .str.upper(), .str.contains(), .str.replace()
- Split strings in pandas
- Combine string cleaning with filtering

---
**Session timing:** 2 hours | DeepSeek assisted ✓

## Why String Cleaning Matters — The City Name Problem

In the Olist customers dataset, **15,606 customers** live in cities whose names contain the word "paulo" — but the city names are stored in inconsistent formats. Some rows say `'sao paulo'`, others say `'São Paulo'`, and some may have extra spaces like `' sao paulo '`. If you tried to count customers per city without cleaning first, Sao Paulo would appear as multiple separate cities and your counts would be wrong.

String cleaning is the bridge between messy real-world text and usable analytical categories. Today you will learn the `.str` accessor — pandas' built-in toolkit for working with text data at scale. With it, you can standardise thousands of city names in a single line of code, search for patterns across an entire column, and extract structured pieces from free-text fields like review comments. These skills are used in every data analysis role.

## Setup — Load the Datasets

In [ ]:
import pandas as pd
from google.colab import drive
import os

drive.mount('/content/drive')
olist_path = '/content/drive/MyDrive/olist-data'

customers    = pd.read_csv(os.path.join(olist_path, 'olist_customers_dataset.csv'))
orders       = pd.read_csv(os.path.join(olist_path, 'olist_orders_dataset.csv'))
reviews      = pd.read_csv(os.path.join(olist_path, 'olist_order_reviews_dataset.csv'))
translations = pd.read_csv(os.path.join(olist_path, 'product_category_name_translation.csv'))

print(f"✓ Loaded {len(customers):,} customers")     # expected: 99,441
print(f"✓ Loaded {len(orders):,} orders")            # expected: 99,441
print(f"✓ Loaded {len(reviews):,} reviews")          # expected: 99,224
print(f"✓ Loaded {len(translations):,} translations") # expected: 71

## Concept 1 — The .str Accessor

In normal Python, if you have a single string you call methods directly on it: `'hello world'.upper()` gives `'HELLO WORLD'`. But in pandas, a column holds thousands of strings at once, and you cannot call `.upper()` on the whole column directly.

The `.str` accessor is pandas' solution. It acts like a "gateway" that lets you apply any standard Python string method to every value in the column simultaneously — as if you had written a `for` loop over every row but in a single, fast, readable line.

Think of it like a remote control: the remote (`.str`) does not do anything by itself, but it lets you operate all the TVs (strings in the column) at once without walking to each one individually.

In [ ]:
# See the raw city names — lowercase, no standardisation
print("Before cleaning:")
print(customers['customer_city'].head(5).tolist())
# expected: ['franca', 'sao bernardo do campo', 'sao paulo', 'mogi das cruzes', 'campinas']

# Clean: strip whitespace then convert to Title Case
customers['customer_city_clean'] = customers['customer_city'].str.strip().str.title()

print("\nAfter cleaning:")
print(customers['customer_city_clean'].head(5).tolist())
# expected: ['Franca', 'Sao Bernardo Do Campo', 'Sao Paulo', 'Mogi Das Cruzes', 'Campinas']

# Count how many customers live in a city containing 'paulo'
paulo_count = customers['customer_city'].str.contains('paulo', case=False).sum()
print(f"\nCustomers in 'paulo' cities: {paulo_count:,}")  # expected: 15,606

## Concept 2 — str.replace() and Category Labels

The product category translation file stores English category names with underscores: `health_beauty`, `computers_accessories`. For display in a report or chart, you want spaces and title case: `Health Beauty`, `Computers Accessories`.

`str.replace('_', ' ')` replaces every underscore with a space across all 71 rows in one call. Chaining `.str.title()` immediately after converts the result to title case. This pattern — chain multiple `.str` operations together — is a signature move of professional pandas data cleaning.

In [ ]:
# Transform category names: underscores → spaces, then title case
print("Before:")
print(translations['product_category_name_english'].head(3).tolist())
# expected: ['health_beauty', 'computers_accessories', 'auto']

translations['category_display'] = (
    translations['product_category_name_english']
    .str.replace('_', ' ')
    .str.title()
)

print("\nAfter:")
print(translations['category_display'].head(3).tolist())
# expected: ['Health Beauty', 'Computers Accessories', 'Auto']

print(f"\nTotal categories: {len(translations)}")  # expected: 71

## Concept 3 — String Length and Null Detection in Reviews

Only some customers leave a written comment with their review. To analyse comment behaviour, you need to (a) detect which rows have a comment and (b) measure how long those comments are.

`str.len()` returns the length of each string — but it returns `NaN` for rows where the string is null, rather than crashing. This makes it a safe way to detect non-null text: if `.str.len()` returns a number, the value exists. This pairs naturally with `.notna()` to create a True/False column marking which reviews include a written message.

In [ ]:
# Add a boolean 'has_comment' column
reviews['has_comment'] = reviews['review_comment_message'].notna()

total = len(reviews)                          # expected: 99,224
with_comment = reviews['has_comment'].sum()   # expected: 40,977
pct = with_comment / total * 100

print(f"Total reviews: {total:,}")             # expected: 99,224
print(f"Reviews with comment: {with_comment:,}") # expected: 40,977
print(f"Percentage with comment: {pct:.1f}%")  # expected: 41.3%

# Measure comment length for those that have one
reviews['comment_length'] = reviews['review_comment_message'].str.len()
print("\nComment length stats (non-null only):")
print(reviews['comment_length'].describe().round(1))
# expected: count 40977.0, mean 103.9

## Using DeepSeek for String Operations

String cleaning tasks are excellent candidates for DeepSeek prompting — the patterns are well-defined and the expected outputs give you a clear way to verify.

**The 4 rules when using DeepSeek:**

1. **Understand first** — describe in plain English what you want before prompting
2. **Prompt with context** — give the DataFrame name, column names, and the specific task
3. **Validate the output** — check against expected values (e.g., paulo_count = 15,606)
4. **Never copy blindly** — explain every line of the generated code

**Example prompt for today's group challenge:**

> *"I have a pandas DataFrame called `orders` with a datetime column `order_purchase_timestamp`. I have another DataFrame called `customers` with a column `customer_city`. I want to merge them on `customer_id`, clean `customer_city` with strip + title case, then count orders per cleaned city and show the top 5. How do I do this?"*

Then verify: Sao Paulo should appear as the top city with 15,540+ orders.

## §4 — Going Deeper: str.contains with regex=False for Speed

By default, `str.contains()` treats the search pattern as a regular expression. That is powerful but slower than a plain string search. For simple lookups where you do not need regex patterns, passing `regex=False` makes the search significantly faster on large datasets.

A subtle gotcha: if your search string accidentally contains regex special characters like `.` or `*`, the default `regex=True` will interpret them as regex operators and give unexpected results. Always use `regex=False` when you are doing a plain literal search.

In [ ]:
# Using regex=False for plain-text search (faster and safer for literals)
sp_count_regex_false = customers['customer_city'].str.contains('sao paulo', case=False, regex=False, na=False).sum()
print(f"Exact 'sao paulo' matches: {sp_count_regex_false:,}")

# Adding na=False prevents NaN values from causing issues
# Without na=False, any null city would cause str.contains to return NaN (not False)
null_cities = customers['customer_city'].isna().sum()
print(f"Null city values: {null_cities}")  # expected: 0 (customers always have a city)

# The 'paulo' cities that contain more than just 'sao paulo'
paulo_cities = customers['customer_city'].str.contains('paulo', case=False, regex=False, na=False)
paulo_city_names = customers[paulo_cities]['customer_city'].str.title().value_counts().head(5)
print("\nTop 'paulo' cities:")
print(paulo_city_names)

## §5 — Common Mistakes

In [ ]:
# ── COMMON MISTAKE 1: Calling string methods directly on a Series ──────────
# WRONG — this raises AttributeError because Series doesn't have .strip() directly:
# customers['customer_city'].strip()  # AttributeError: 'Series' object has no attribute 'strip'

# CORRECT — use the .str accessor:
sample_clean = customers['customer_city'].str.strip().str.title()
print("First 3 cleaned cities:", sample_clean.head(3).tolist())

# ── COMMON MISTAKE 2: str.replace changes the regex by default ───────────────
# WRONG — this replaces nothing because '.' matches any character in regex:
# translations['category_display'] = translations['product_category_name_english'].str.replace('.', ' ')
# (replaces EVERY character with a space)

# CORRECT — use regex=False for literal replacement:
safe_replace = translations['product_category_name_english'].str.replace('_', ' ', regex=False).str.title()
print("\nFirst 3 safe categories:", safe_replace.head(3).tolist())
# expected: ['Health Beauty', 'Computers Accessories', 'Auto']

## §6 — Mini-Challenge

The `reviews` DataFrame has a `review_comment_message` column. Some comments are very long.

**Your task:**
1. Add `comment_length` column: length of `review_comment_message` (null for rows with no comment)
2. Filter to only reviews WITH a comment (`comment_length.notna()`)
3. Find the 3 longest comments by `comment_length`
4. Print the length of the longest comment

**Expected:** the longest comment is several hundred characters long.

⏱ ~5 minutes

In [ ]:
# Mini-Challenge — given variables (do not change)
# reviews DataFrame is already loaded above

# Step 1 — add comment_length column
# reviews['comment_length'] = reviews['review_comment_message'].str.len()

# Step 2 — filter to reviews with a comment
# reviews_with_comments = reviews[reviews['comment_length'].notna()]

# Step 3 — find the 3 longest comments
# longest_3 = reviews_with_comments.nlargest(3, 'comment_length')[['review_comment_message', 'comment_length']]

# Step 4 — print the length of the longest comment
# print(longest_3['comment_length'].max())

## §7 — Group Exercise (35 minutes)

Work in your groups. Verify each expected output before moving to the next task.

**Task 1:** Load customers. Clean `customer_city`: strip whitespace + title case. How many unique cities are there after cleaning? *(May differ slightly from before due to space normalisation)*

**Task 2:** Load reviews. Add `has_comment` column = True/False based on whether `review_comment_message` is not null. What percentage of reviews have a comment?  
*Expected: (99,224 − 58,247) / 99,224 = 41.3%*

**Task 3:** Load the translation file. Add `category_display` column: English category name with underscores replaced by spaces, in Title Case. Print all 71 display names.

**Task 4 (DeepSeek challenge):** Ask DeepSeek to find the top 5 cities by number of orders. You will need to merge `orders` with `customers` on `customer_id`, clean `customer_city`, then groupby and count. Verify: **Sao Paulo is #1 with 15,540+ orders.**

In [ ]:
# Group Exercise

# Task 1 — unique cities after cleaning
customers_clean = customers.copy()
customers_clean['customer_city_clean'] = customers_clean['customer_city'].str.strip().str.title()
# unique_cities_before = customers['customer_city'].nunique()
# unique_cities_after = customers_clean['customer_city_clean'].nunique()
# print(f"Unique cities before: {unique_cities_before}, after: {unique_cities_after}")

# Task 2 — % reviews with a comment
# reviews['has_comment'] = reviews['review_comment_message'].notna()
# pct_with_comment = reviews['has_comment'].mean() * 100
# print(f"Reviews with comment: {pct_with_comment:.1f}%")  # expected: ~41.3%

# Task 3 — category display names
# translations['category_display'] = translations['product_category_name_english'].str.replace('_', ' ', regex=False).str.title()
# print(translations['category_display'].tolist())  # 71 names

# Task 4 (DeepSeek challenge) — top 5 cities by orders
# (ask DeepSeek, then verify Sao Paulo = #1 with 15,540+)

## §8 — Session Summary

| Concept | Key Syntax | Example |
|---|---|---|
| Clean whitespace + case | `.str.strip().str.title()` | `'sao paulo'` → `'Sao Paulo'` |
| Search in strings | `.str.contains('paulo', case=False)` | Returns True/False Series |
| Replace characters | `.str.replace('_', ' ', regex=False)` | `'health_beauty'` → `'health beauty'` |
| String length | `.str.len()` | Returns int or NaN for null |
| Detect non-null text | `.notna()` | Returns True/False per row |
| Chain operations | `.str.method1().str.method2()` | Clean and format in one line |

---
**Weekly Assignment:** Submit `week6_assignment.ipynb` covering all four tasks from the assignment sheet. The key verified outputs are: median delivery = 10.0 days, longest delivery = 209 days, Sao Paulo = 15,540 customers.

**Coming up next week — Week 7:** Merging DataFrames — using `pd.merge()` to combine orders, customers, and products into a single analysis-ready table.